In [1]:
import pandas as pd
import glob
import os

In [ ]:
# Load each semester's grade distribution file
path = '/content/'
all_files = glob.glob(os.path.join(path, "*.csv"))

In [3]:
# Read each CSV into a list of DataFrames
df_list = []
for file in all_files:
    df = pd.read_csv(file)
    df_list.append(df)

In [4]:
# Concatenate all DataFrames into one master DataFrame
master_df = pd.concat(df_list, ignore_index=True)

Ensure consistency with the columns.

In [6]:
# Rename columns with special characters manually
master_df.rename(columns={
    'INC/NA': 'INC',
    'A+': 'A_PLUS',
    'A-': 'A_MINUS',
    'B+': 'B_PLUS',
    'B-': 'B_MINUS',
    'C+': 'C_PLUS',
    'C-': 'C_MINUS',
    'D+': 'D_PLUS'
}, inplace=True)

# Rename the ubiquitous language names to be more clear
master_df.rename(columns={
    'subject': 'course_subject',
    'nbr': 'course_number',
    'section': 'course_section',
    'prof': 'prof_name',
    'total': 'total_enrollment'
}, inplace=True)

# Convert all column names to lowercase and replace spaces with underscores
master_df.columns = master_df.columns.str.lower().str.replace(' ', '_')

# Print the new columns to verify they look clean
print(master_df.columns)

Index(['term', 'course_subject', 'course_number', 'course_name',
       'course_section', 'prof_name', 'total_enrollment', 'a_plus', 'a',
       'a_minus', 'b_plus', 'b', 'b_minus', 'c_plus', 'c', 'c_minus', 'd_plus',
       'd', 'f', 'w', 'inc', 'avg_gpa'],
      dtype='object')


## Data Preview



In [7]:
# Reorder columns visually
preferred_order = [
    'term', 'course_subject', 'course_number', 'course_name', 'course_section', 'prof_name', 'total_enrollment',
    'a_plus', 'a', 'a_minus', 'b_plus', 'b', 'b_minus', 'c_plus', 'c',
    'c_minus', 'd_plus', 'd', 'f', 'w', 'inc', 'avg_gpa'
]

master_df = master_df[preferred_order]

In [28]:
# Preview the data to ensure it merged correctly
display(master_df.head())
display(master_df.tail())

,term,course_subject,course_number,course_name,course_section,prof_name,total_enrollment,a_plus,a,a_minus,...,b_minus,c_plus,c,c_minus,d_plus,d,f,w,inc,avg_gpa
0,Spring 2025,ACCT,100,Fin & Mgr Acct,1,"HO, V",10,0,4,1,...,0,1,2,0,0,0,0,0,0,3.260
1,Spring 2025,ACCT,101,Intro Thry & Prac of Acct I,4,"DAUBER, N",40,2,11,8,...,2,0,1,0,0,1,0,7,0,3.491
2,Spring 2025,ACCT,101,Intro Thry & Prac of Acct I,6,"RUTHIZER, S",34,1,6,4,...,1,3,2,0,0,0,0,3,0,3.213
3,Spring 2025,ACCT,101,Intro Thry & Prac of Acct I,3,"DAUBER, N",40,1,5,7,...,7,2,4,2,0,0,1,2,0,2.974
4,Spring 2025,ACCT,101,Intro Thry & Prac of Acct I,13,"FITZSIMONS, S",21,0,2,1,...,4,0,0,2,0,0,0,1,2,2.939


,term,course_subject,course_number,course_name,course_section,prof_name,total_enrollment,a_plus,a,a_minus,...,b_minus,c_plus,c,c_minus,d_plus,d,f,w,inc,avg_gpa
24630,Fall 2024,WGS,101W,Intro Women & Gender Studies,4,"REECE, K",26,4,4,16,...,0,0,0,0,0,0,0,1,0,3.780
24631,Fall 2024,WGS,101W,Intro Women & Gender Studies,4,"REECE, K",26,4,4,16,...,0,0,0,0,0,0,0,1,0,3.780
24632,Fall 2024,WGS,101W,Intro Women & Gender Studies,2,"GIARDINA, C",25,4,2,3,...,0,0,0,1,0,0,1,3,0,3.255
24633,Fall 2024,WGS,101W,Intro Women & Gender Studies,3,"GIARDINA, C",25,0,3,2,...,2,0,3,0,0,0,0,5,1,3.121
24634,Fall 2024,WGS,201W,Theories of Feminism,1,"SAN EMETERIO, G",25,12,5,3,...,1,0,0,0,0,0,0,3,1,3.895


## Data Cleaning

In [29]:
# Print every column name and the count of missing (NaN) values in it
print(master_df.isna().sum())

term                0
course_subject      0
course_number       0
course_name         0
course_section      0
prof_name           0
total_enrollment    0
a_plus              0
a                   0
a_minus             0
b_plus              0
b                   0
b_minus             0
c_plus              0
c                   0
c_minus             0
d_plus              0
d                   0
f                   0
w                   0
inc                 0
avg_gpa             0
dtype: int64


Ensure all grade-related and professor columns don't contain NaNs.



In [10]:
# List of all grade-related columns
grade_cols = [
    'a_plus', 'a', 'a_minus', 'b_plus', 'b', 'b_minus',
    'c_plus', 'c', 'c_minus', 'd_plus', 'd', 'f', 'w', 'inc'
]

# Fill NaNs with 0 for all these columns. Currently, only d_plus contains NaNs.
master_df[grade_cols] = master_df[grade_cols].fillna(0)

# Forces the 'd_plus' column to be of type int
master_df['d_plus'] = master_df['d_plus'].astype(int)

In [11]:
# Drops any row where the 'prof_name' column is NaN
master_df = master_df.dropna(subset=['prof_name'])

Ensure all numeric columns contain only numerical values and are non-negative.

In [12]:
# Explicitly define the columns that MUST be numeric
numeric_cols = [
    'total_enrollment', 'a_plus', 'a', 'a_minus', 'b_plus', 'b', 'b_minus',
    'c_plus', 'c', 'c_minus', 'd_plus', 'd', 'f', 'w', 'inc', 'avg_gpa'
]

In [16]:
# Force conversion to find hidden strings
coerced_df = master_df[numeric_cols].apply(pd.to_numeric, errors='coerce')
invalid_mask = coerced_df.isna() & master_df[numeric_cols].notna()
invalid_count = invalid_mask.sum()

# Report the findings
print("Count of invalid string/characters per column:")
print(invalid_count[invalid_count > 0])

# If clean, permanently update master_df to bypass safety checks later
if invalid_count.sum() == 0:
    master_df[numeric_cols] = coerced_df
    print("\nAll columns successfully verified and converted to pure numeric types.")

Count of invalid string/characters per column:
Series([], dtype: int64)

All columns successfully verified and converted to pure numeric types.


Running the previous cell the first time identified that the grade column c had one record where its value was a invalid string. We must drop that one invalid row.

In [17]:
# Create a mask specifically for the bad row in column 'c'
bad_c_mask = pd.to_numeric(master_df['c'], errors='coerce').isna() & master_df['c'].notna()

# Isolate and display the corrupted record so you can see the culprit
bad_record = master_df[bad_c_mask]
print("Here is the corrupted record:")
display(bad_record[['term', 'prof_name', 'course_subject', 'course_number', 'c']])

Here is the corrupted record:


,term,prof_name,course_subject,course_number,c


Proceed to drop the row where the 'c' column contains 'ra'.

In [18]:
# Drop the corrupted row from the main DataFrame by keeping everything EXCEPT the mask (~)
master_df = master_df[~bad_c_mask]
print(f"\nNew DataFrame shape: {master_df.shape}")
# Convert the columns, since the bad data is gone
master_df[numeric_cols] = master_df[numeric_cols].apply(pd.to_numeric, errors='coerce')


New DataFrame shape: (24575, 22)


Ensure the numeric columns are non-negative.

In [19]:
# Check if the numeric columns are non-negative
negatives_count = (master_df[numeric_cols] < 0).sum()

print("Count of negative values per column:")
print(negatives_count[negatives_count > 0])

Count of negative values per column:
Series([], dtype: int64)


Ensure total_enrollment is a positive integer value.

In [20]:
# Isolate rows where total_enrollment is 0 or less
invalid_enrollment = master_df[master_df['total_enrollment'] <= 0]

# Count them
print(f"Rows with 0 or negative enrollment: {len(invalid_enrollment)}")

# Display the first few rows so you can inspect them visually
if len(invalid_enrollment) > 0:
    print("\nSample of invalid enrollment rows:")
    display(invalid_enrollment[['term', 'course_subject', 'course_number', 'total_enrollment']].head(10))

Rows with 0 or negative enrollment: 0


Ensure consistent formatting across the **term** column.

In [21]:
import re

def standardize_term(term_string):
    term = str(term_string).upper().replace(' ', '')

    # Extract the year (looks for 2 to 4 digits)
    year_match = re.search(r'\d{2,4}', term)
    if not year_match:
        return term_string

    year = year_match.group()
    if len(year) == 2:
        year = '20' + year

    # Extract the season
    if 'S' in term and 'F' not in term:
        season = 'Spring'
    elif 'F' in term:
        season = 'Fall'
    else:
        season = 'Unknown'

    return f"{season} {year}"

# Apply the function to the lowercase 'term' column
master_df['term'] = master_df['term'].apply(standardize_term)

# Verify the changes
print(master_df['term'].unique())

['Spring 2025' 'Fall 2022' 'Spring 2023' 'Spring 2024' 'Spring 2022'
 'Fall 2025' 'Fall 2023' 'Fall 2021' 'Fall 2024']


Ensure every professor's name must follow the *Last Name, First Initial* format.

In [24]:
# Regex pattern: Letters/spaces/hyphens, followed by a comma, a space, and a single uppercase letter
pattern = r'^[A-Za-z\s\-\'’]+, [A-Z]$'

# Find all rows that DO NOT match the pattern
invalid_profs = master_df[~master_df['prof_name'].str.match(pattern, na=False)]

# Print unique invalid names to review them
print("Non-conforming professor names:")
print(invalid_profs['prof_name'].unique())

Non-conforming professor names:
[]


Running the previous cell the first time produced: ['0' 'ABREGO JR., R']. We proceed to drop the corrupted '0' row and strip suffixes from the names.

In [23]:
# Drop the corrupted '0' row
master_df = master_df[master_df['prof_name'] != '0']

# Strip 'JR.' or 'SR.' (and similar suffixes) from the names
master_df['prof_name'] = master_df['prof_name'].str.replace(r'\s+(JR\.|SR\.)', '', regex=True)

Examine the rows/sections where the avg_gpa column contains NaNs.

In [25]:
# Isolate the rows where avg_gpa is missing
nan_gpa_rows = master_df[master_df['avg_gpa'].isna()].copy()

# Print how many there are to see the scale of the edge case
print(f"Total rows missing GPA: {len(nan_gpa_rows)}")

# Look at the letter grade columns for the all flagged rows
# to confirm they are mostly W's, INC's, or Pass/Fail scenarios
print(nan_gpa_rows[['course_subject', 'course_number', 'total_enrollment', 'a', 'f', 'w', 'inc']])

Total rows missing GPA: 41
     course_subject course_number  total_enrollment   a  f  w  inc
2889          URBST           265                 1   0  0  0    0
2890          URBST           266                12   3  0  2    0
2891          URBST           266                12   3  0  2    0
2892          URBST           273                19   2  0  0    0
2893          URBST           307                 5   0  0  1    0
2894          URBST           328                17  10  1  2    0
2895          URBST           370                12   0  0  0    0
2896          URBST           372                 4   1  0  0    0
2897          URBST           708                 3   3  0  0    0
2898          URBST           721                 8   1  0  0    0
2899          URBST           724                 9   7  0  0    1
2900          URBST           725                19  10  0  0    0
2901          URBST           728                10   8  0  0    0
2902          URBST           791  

Initially, the reasons why I wanted keep the rows/sections where the **avg_gpa** column was NaN were because these cases were possible:
- Certain classes offered students the option to base their grade on the Pass/Fail system
- All of the students for a certain class earned a W grade due to withdrawing

However, after the running the check above, I saw that the students enrolled in these sections (course_subjects: URBST and WGS) definitely received letter grades, which means there absolutely should be an avg_gpa attached to these rows. Since only 41 rows were flagged containing an **avg_gpa** column value of NaN, I wanted to see if I could manually compute the missing avg GPAs by running a check first. The code cell below checks to see if each one of these rows has a total_enrollment count that accounts for every student's assigned grade.

In [26]:
# 2. Define ALL columns that account for a student's final status
all_grade_cols = [
    'a_plus', 'a', 'a_minus', 'b_plus', 'b', 'b_minus',
    'c_plus', 'c', 'c_minus', 'd_plus', 'd', 'f', 'w', 'inc'
]

# 3. Sum these columns across the row (axis=1) to get the "actual" student count
nan_gpa_rows['calculated_enrollment'] = nan_gpa_rows[all_grade_cols].sum(axis=1)

# 4. Create a boolean flag to see if the math perfectly matches
nan_gpa_rows['is_perfect_match'] = nan_gpa_rows['calculated_enrollment'] == nan_gpa_rows['total_enrollment']

# 5. Print a high-level summary
print("--- Validation Summary ---")
print(nan_gpa_rows['is_perfect_match'].value_counts())
print("-" * 26)

# 6. Display the rows that FAILED the match so we can inspect the damage
failed_rows = nan_gpa_rows[~nan_gpa_rows['is_perfect_match']]

if len(failed_rows) > 0:
    print(f"⚠️ Found {len(failed_rows)} rows where grades do NOT add up to total_enrollment:")
    display_cols = ['course_subject', 'course_number', 'total_enrollment', 'calculated_enrollment'] + all_grade_cols
    display(failed_rows[display_cols])
else:
    print("SUCCESS: All 41 rows have perfect grade-to-enrollment alignment!")

--- Validation Summary ---
is_perfect_match
True    41
Name: count, dtype: int64
--------------------------
SUCCESS: All 41 rows have perfect grade-to-enrollment alignment!


Given that the above check confirms that each one of these rows has a perfect grade-to-enrollment alignment, we can proceed to compute the missing avg GPAs.

In [27]:
import numpy as np

# Define the standard Queens College grade weights
grade_weights = {
    'a_plus': 4.0, 'a': 4.0, 'a_minus': 3.7,
    'b_plus': 3.3, 'b': 3.0, 'b_minus': 2.7,
    'c_plus': 2.3, 'c': 2.0, 'c_minus': 1.7,
    'd_plus': 1.3, 'd': 1.0, 'f': 0.0
}

# Calculate the total grade points for every row
total_grade_points = sum(master_df[grade] * weight for grade, weight in grade_weights.items())

# Calculate how many students actually factor into the GPA (ignoring W and INC)
gpa_students = master_df[list(grade_weights.keys())].sum(axis=1)

# Compute the reconstructed GPA (using np.where to prevent division-by-zero for all-W classes)
reconstructed_gpa = np.where(gpa_students > 0, total_grade_points / gpa_students, np.nan)

# Fill ONLY the NaN values in avg_gpa column and round to 3 decimal places to match standard GPA formatting
master_df['avg_gpa'] = master_df['avg_gpa'].fillna(pd.Series(reconstructed_gpa, index=master_df.index)).round(3)

# Verify the update worked
remaining_nans = master_df['avg_gpa'].isna().sum()
print(f"Update complete! Remaining rows missing GPA: {remaining_nans}")

Update complete! Remaining rows missing GPA: 0


## Exporting Data

In [31]:
# Export the merged data to a new master CSV
master_df.to_csv('master_grade_distribution.csv', index=False)